# 实验报告 03：正则表达式与网络结构化数据

**姓名：** 金晨曦　　**学号：** 20244081111　　**日期：** 2026-11-18

**运行环境：** conda 虚拟环境 `dpenv`（Python 3.11，BeautifulSoup 4.15）。

本报告按 PDF 中的实验 1–3 顺序完成：正则表达式基础匹配、分组/回溯引用/替换、BeautifulSoup 解析 HTML。所有代码单元按顺序执行，关键结果直接输出。

## 一、身份与环境记录

按题目要求，先在第一个代码单元填写自己的学号，再依次运行后续代码单元。

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244081111

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] dpenv
[学号]       20244081111
[机器指纹] 504894f62c9f33b0
[RUN_ID]   20244081111-504894f62c9f33b0
[Python]    3.11.17


In [2]:
import re
from bs4 import BeautifulSoup

print("re 模块已导入；BeautifulSoup 版本:", __import__("bs4").__version__)

re 模块已导入；BeautifulSoup 版本: 4.15.0


## 实验 1：正则表达式的基础匹配

### 1. 数字、空白与单词边界

在 `"cat on mat; later: cats and dogs"` 上分别使用 `\d+`、`\s+` 和 `\bcat\b`。

In [3]:
text = "cat on mat; later: cats and dogs"

print("原文本:", text)
print(r"\d+ 的匹配结果:", re.findall(r"\d+", text))
print(r"\s+ 的匹配结果:", re.findall(r"\s+", text))
print(r"\bcat\b 的匹配结果:", re.findall(r"\bcat\b", text))

原文本: cat on mat; later: cats and dogs
\d+ 的匹配结果: []
\s+ 的匹配结果: [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b 的匹配结果: ['cat']


**匹配规则说明：**

- `\d+` 匹配一个或多个数字。原文本中没有数字，所以结果为空列表。
- `\s+` 匹配一个或多个空白字符。文本中共有 6 个空格，每个空格单独构成一次匹配。
- `\bcat\b` 中 `\b` 是单词边界，要求 `cat` 前后都不是单词字符。文本中的 `cats` 在 `t` 后还有 `s`，不构成右边界，所以只匹配到独立的 `cat`。

### 2. 选择分支与匹配顺序

在 `"cat dog bird CAT"` 上使用 `cat|dog`；并比较 `re.findall(r"a|aa|aaa", "aaaa")` 与 `re.findall(r"aaa|aa|a", "aaaa")`。

In [4]:
text = "cat dog bird CAT"

print('re.findall(r"cat|dog", text):', re.findall(r"cat|dog", text))
print('re.findall(r"a|aa|aaa", "aaaa"):', re.findall(r"a|aa|aaa", "aaaa"))
print('re.findall(r"aaa|aa|a", "aaaa"):', re.findall(r"aaa|aa|a", "aaaa"))

re.findall(r"cat|dog", text): ['cat', 'dog']
re.findall(r"a|aa|aaa", "aaaa"): ['a', 'a', 'a', 'a']
re.findall(r"aaa|aa|a", "aaaa"): ['aaa', 'a']


**匹配规则说明：**

- `cat|dog` 是“或”关系，正则默认区分大小写，所以 `CAT` 不会被匹配，结果为 `['cat', 'dog']`。
- `a|aa|aaa` 按从左到右的顺序尝试分支：在 `"aaaa"` 的每个位置先尝试 `a` 就成功，因此得到 4 个 `a`。
- `aaa|aa|a` 同样按书写顺序尝试：第一个位置先尝试 `aaa` 成功，剩下一个 `a` 再匹配 `a`，结果为 `['aaa', 'a']`。这说明选择分支是“最左优先”，不是“最长优先”；若要最长匹配，应把长分支写在前面。

### 3. `*`、`+`、`?`、`{m}`、`{m,n}` 量词

在 `"ct cat caat caaat"` 上分别使用 `ca*t`、`ca+t`、`ca?t`、`ca{2}t` 和 `ca{1,2}t`。

In [5]:
text = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

for pattern in patterns:
    print(f"{pattern:10} -> {re.findall(pattern, text)}")

ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


**量词含义与结果说明：**

- `ca*t`：`a*` 表示 0 个或多个 `a`，所以 `ct`（0 个）、`cat`（1 个）、`caat`（2 个）、`caaat`（3 个）都匹配。
- `ca+t`：`a+` 表示 1 个或多个 `a`，所以不匹配 `ct`，其余三个都匹配。
- `ca?t`：`a?` 表示 0 个或 1 个 `a`，只匹配 `ct` 和 `cat`；`caat`、`caaat` 中 `c` 后面连续两个以上 `a`，无法在 `t` 前结束。
- `ca{2}t`：恰好 2 个 `a`，只匹配 `caat`。
- `ca{1,2}t`：1 到 2 个 `a`，匹配 `cat` 和 `caat`；`caaat` 有 3 个 `a`，超出上限，不匹配。

## 实验 2：分组、回溯引用与替换

### 1. 用捕获组提取邮箱用户名和域名

在 `"联系：alice@example.com；备选：bob@test.org"` 中提取每个邮箱的用户名和域名。

In [6]:
text = "联系：alice@example.com；备选：bob@test.org"
pattern = r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})"

matches = re.findall(pattern, text)
for i, (user, domain) in enumerate(matches, 1):
    print(f"邮箱 {i}: 用户名 = {user!r}, 域名 = {domain!r}")

邮箱 1: 用户名 = 'alice', 域名 = 'example.com'
邮箱 2: 用户名 = 'bob', 域名 = 'test.org'


**匹配规则说明：** 第一个括号 `([A-Za-z0-9._%+-]+)` 捕获 `@` 前的用户名，第二个括号 `([A-Za-z0-9.-]+\.[A-Za-z]{2,})` 捕获 `@` 后的域名（要求至少有一个点号和 2 个以上字母的顶级域名）。当正则中有多个捕获组时，`re.findall()` 返回由各组成员组成的元组列表。

### 2. 用后向引用查找连续重复的单词

在 `"cat cat, dog cat, bird bird"` 中找出连续重复的单词。

In [7]:
text = "cat cat, dog cat, bird bird"
pattern = r"\b(\w+)\s+\1\b"

print("重复的单词:", re.findall(pattern, text))
for m in re.finditer(pattern, text):
    print(f"  完整匹配: {m.group(0)!r}，位置 {m.start()}–{m.end() - 1}")

重复的单词: ['cat', 'bird']
  完整匹配: 'cat cat'，位置 0–6
  完整匹配: 'bird bird'，位置 18–26


**匹配规则说明：** `(\w+)` 捕获第一个单词，`\s+` 匹配中间的一个或多个空白，`\1` 是后向引用，要求第二处文字与第一组捕获的内容完全相同，最后的 `\b` 保证单词边界。因此 `cat cat` 和 `bird bird` 被匹配，而 `dog cat` 两个单词不同，不会被匹配。`re.findall()` 只返回捕获组内容（这里是第一个单词）。

### 3. 日期格式转换

将 `"2024-05-01\n2024/05/02\n2024.05.03"` 从“年-月-日”转换为“月-日-年”，保留原来的行数。

In [8]:
text = "2024-05-01\n2024/05/02\n2024.05.03"
pattern = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"

converted = re.sub(pattern, r"\2-\3-\1", text)
print("转换前:")
print(text)
print("\n转换后:")
print(converted)
print("\n行数:", len(text.splitlines()), "->", len(converted.splitlines()))

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后:
05-01-2024
05-02-2024
05-03-2024

行数: 3 -> 3


**匹配规则说明：** `(\d{4})` 捕获四位年份，`[-/.]` 匹配三种分隔符中的任意一种，后两个 `(\d{2})` 分别捕获月份和日期。替换串 `\2-\3-\1` 按“月-日-年”重新排列三个捕获组。`re.sub()` 只替换匹配到的日期部分，不改变换行符，所以转换前后都是 3 行。

### 4. 电话号码格式统一

将 `"联系我：123-456-7890 或 987.654.3210"` 中的电话号码统一为 `(123) 456-7890` 形式。

In [9]:
text = "联系我：123-456-7890 或 987.654.3210"
pattern = r"(\d{3})[-.](\d{3})[-.](\d{4})"

print("统一后:", re.sub(pattern, r"(\1) \2-\3", text))

统一后: 联系我：(123) 456-7890 或 (987) 654-3210


**匹配规则说明：** 三个捕获组分别对应区号、中间三位和后四位，`[-.]` 同时兼容连字符和点号两种分隔符。替换串 `(\1) \2-\3` 在区号外加圆括号，并把中间和后四位用连字符连接；原文本中的中文说明文字不受影响。

## 实验 3：使用 BeautifulSoup 解析 HTML

使用题目给出的 HTML 字符串，不依赖网页下载。

In [10]:
html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

### 1. 构建解析树，输出标题与 `h1`

用 `BeautifulSoup(html_doc, "html.parser")` 构建解析树，输出页面标题文本、`h1` 的标签名与文本。

In [11]:
soup = BeautifulSoup(html_doc, "html.parser")

print("页面标题文本:", soup.title.get_text())
h1 = soup.h1
print("h1 标签名:", h1.name)
print("h1 文本:", h1.get_text())

页面标题文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


**说明：** `soup.title` 取得 `<title>` 标签，`get_text()` 取出其中的纯文本“实验课程”；`h1.name` 返回标签名 `"h1"`，`h1.get_text()` 返回“数据分析”。

### 2. `find_all()` 查找段落与链接

使用 `find_all()` 找出所有 `class="course"` 的段落和所有链接，逐项输出链接文字与 `href` 属性。

In [12]:
course_ps = soup.find_all("p", class_="course")
links = soup.find_all("a")

print("class=course 的段落数:", len(course_ps))
for i, p in enumerate(course_ps, 1):
    print(f"  段落 {i}: {p.get_text()}")

print("\n链接数:", len(links))
for i, a in enumerate(links, 1):
    print(f"  链接 {i}: 文字 = {a.get_text()!r}, href = {a.get('href')!r}")

class=course 的段落数: 2
  段落 1: 第六讲：正则表达式
  段落 2: 第七讲：结构化数据

链接数: 2
  链接 1: 文字 = '讲义六', href = 'https://example.org/lecture06'
  链接 2: 文字 = '讲义七', href = 'https://example.org/lecture07'


**说明：** `find_all("p", class_="course")` 按标签名和 class 属性筛选，得到两个课程段落；`find_all("a")` 得到两个链接。链接文字用 `get_text()` 读取，目标地址用属性访问 `a.get("href")` 读取，避免把标签本身打印出来。

### 3. `get_text()` 提取可见文本

用 `get_text()` 提取页面可见文本，比较整个文档、单个段落和单个链接的文本结果。

In [13]:
print("整个文档 get_text():")
print(repr(soup.get_text()))
print("\n单个段落 get_text():", repr(course_ps[0].get_text()))
print("单个链接 get_text():", repr(links[0].get_text()))

整个文档 get_text():
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

单个段落 get_text(): '第六讲：正则表达式'
单个链接 get_text(): '讲义六'


**说明：** `soup.get_text()` 会递归拼接整个文档中所有标签的文本，包含 `<title>`、`<h1>`、两个课程段落和两个链接文字，并保留源码中的换行与缩进空白，所以结果中会出现多个 `\n` 和空格。对单个 `Tag` 调用 `get_text()` 时只提取该标签内部的文本：段落返回“第六讲：正则表达式”，链接返回“讲义六”，不包含 URL 或外层页面文字。

## 小结

- 实验 1 练习了字符类、量词、单词边界和选择分支，重点理解了 `\b` 的边界含义以及选择分支“最左优先”的匹配顺序。
- 实验 2 使用捕获组提取邮箱字段、用 `\1` 后向引用查找重复单词，并用 `re.sub()` 完成日期和电话号码的格式化替换。
- 实验 3 使用 BeautifulSoup 构建解析树，练习了标签/属性查找、链接属性读取以及 `get_text()` 的局部与整体文本提取。